<a href="https://colab.research.google.com/github/Saiful-2/house-price-prediction/blob/main/notebooks/10_housing_final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔹 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 🔹 2. Load Dataset

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/Saiful-2/datasets/main/ames_housing_cleaned.csv")

df.head()

,ms_subclass,ms_zoning,lot_frontage,lot_area,street,alley,lot_shape,land_contour,utilities,lot_config,...,pool_area,pool_qc,fence,misc_feature,misc_val,mo_sold,yr_sold,sale_type,sale_condition,saleprice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,Ex,NaN,Shed,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,Ex,MnPrv,Shed,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,Ex,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,Ex,NaN,Shed,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,Ex,MnPrv,Shed,0,3,2010,WD,Normal,189900


# 🔹 3. Define Features & Target

In [3]:
target = "log_saleprice"
df[target] = np.log(df["saleprice"])

X = df.drop(columns=[target])
y = df[target]

# 🔹 4. Train-Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🔹 5. Preprocessing

In [5]:
from sklearn.impute import SimpleImputer

categorical_cols = X.select_dtypes(include='object').columns
numerical_cols = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

# 🔹 6. Final Model Pipeline

👉 Use the best model from evaluation (Gradient Boosting)

In [6]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    ))
])

# 🔹 7. Train Final Model

In [7]:
final_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  Index(['ms_subclass', 'lot_frontage', 'lot_area', 'overall_qual',
       'overall_cond', 'year_built', 'year_remodadd', 'mas_vnr_area',
       'bsmtfin_sf_1', 'bsmtfin_sf_2', 'bsmt_unf_sf', 'total_bsmt_sf',
       '1st_flr_sf', '2nd_flr_sf', 'low_qual_fin_sf', 'gr_liv_area',
       'bsmt_full_bath', 'bsmt_hal...
       'bsmtfin_type_1', 'bsmtfin_type_2', 'heating', 'heating_qc',
       'central_air', 'electrical', 'kitchen_qual', 'functional',
       'fireplace_qu', 'garage_type', 'garage_finish', 'garage_qual',
       'garage_cond', 'paved_drive', 'pool_qc', 'fence', 'misc_feature',
       'sale_type', 'sale_condition'],
      dtype='object'))])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, n_estimators=200,
                                           random_state=42))])

# 🔹 8. Final Evaluation (Sanity Check)

In [8]:
preds = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("Final Model RMSE:", rmse)
print("Final Model R2:", r2)

Final Model RMSE: 0.002983055529356198
Final Model R2: 0.999948508513628


# 🔹 9. Train on FULL DATA

In [9]:
final_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  Index(['ms_subclass', 'lot_frontage', 'lot_area', 'overall_qual',
       'overall_cond', 'year_built', 'year_remodadd', 'mas_vnr_area',
       'bsmtfin_sf_1', 'bsmtfin_sf_2', 'bsmt_unf_sf', 'total_bsmt_sf',
       '1st_flr_sf', '2nd_flr_sf', 'low_qual_fin_sf', 'gr_liv_area',
       'bsmt_full_bath', 'bsmt_hal...
       'bsmtfin_type_1', 'bsmtfin_type_2', 'heating', 'heating_qc',
       'central_air', 'electrical', 'kitchen_qual', 'functional',
       'fireplace_qu', 'garage_type', 'garage_finish', 'garage_qual',
       'garage_cond', 'paved_drive', 'pool_qc', 'fence', 'misc_feature',
       'sale_type', 'sale_condition'],
      dtype='object'))])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, n_estimators=200,
                                           random_state=42))])

# 💾 🔹 10. Save Model (Pipeline)

In [10]:
joblib.dump(final_model, "ames_final_model.pkl")

print("✅ Model saved successfully!")

✅ Model saved successfully!


# 🔹 11. Save Feature Columns (IMPORTANT for UI)

In [11]:
feature_columns = X.columns.tolist()

joblib.dump(feature_columns, "feature_columns.pkl")

['feature_columns.pkl']

# 🔹 12. Test Loading

In [12]:
loaded_model = joblib.load("ames_final_model.pkl")

sample = X.iloc[[0]]
prediction = loaded_model.predict(sample)

print("Sample Prediction:", prediction)

Sample Prediction: [12.27806462]


# 🔹 13. Convert Prediction Back to Original Price

In [13]:
def predict_price(model, input_df):
    log_price = model.predict(input_df)
    return np.exp(log_price)

real_price = predict_price(loaded_model, sample)
print("Predicted Sale Price:", real_price)

Predicted Sale Price: [214929.34349481]


# 🔹 14. Create Inference Function (FOR STREAMLIT)

In [14]:
def predict_house_price(input_dict):
    df_input = pd.DataFrame([input_dict])

    log_price = loaded_model.predict(df_input)
    price = np.exp(log_price)

    return float(price[0])

# 🔹 15. Example Input

In [15]:
sample_input = X.iloc[0].to_dict()

predict_house_price(sample_input)

214929.34349480743